(Auteur et concepteur : Samuel Desnoyers, 2025)

Ce code sert à détecter les affiliations de recherche dans des chaînes textuels.

Deux fichiers Excel sont nécessaires
1- un fichier "dictionnaire" : les données d'affiliations de nos auteurs
2- un fichier "data" : dont les données sont tirées de notre fichier de conversion officiel


Le fichier "dictionnaire" doit contenir deux champs:
1- ror : qui correspond à la colonne "ID unique (ROR ou textuel)" du fichier de conversion
2- nom : qui correspond à la colonne "affiliation" du fichier de conversion

Le fichier "data" doit contenir deux champs:
1- id : un id unique qui permettra de rapporter correctement les résultats (par exemple : no_article.no_auteur.index_auteur)
2- author_affiliation_content : ce champ vient directement des données extraites sur redevances et contient les affiliations


Les deux premières cellules du programme normalisent le texte dans ces deux fichiers:
- minuscule
- remplacement de certains termes
- suppression de mots vides
- retrait des caractères accentués et de la ponctuation

La dernière cellule tente de retrouver un match dans le fichier de conversion à partir des données d'affiliation.

Le programme retourne trois fichiers:
1- data_clean : le fichier data normalisé
2- dictionnaire_clean: le fichier dictionnaire normalisé
3- results : les résultats à utiliser

Les résultats qui nous intéressent sont dans la dernière colonne de ce fichier. Grâce à l'ID, ces résultats peuvent être rapportés sur les données d'origine.




Temps d'exécution pour 150 000 lignes dans le fichier data = 3h


In [4]:
import pandas as pd
from unidecode import unidecode
import re

# NETTOYAGE DES ALIAS

# Charger le fichier Excel
fichier = "dictionnaire.xlsx"
df = pd.read_excel(fichier)

# Supprimer les lignes doublons
df = df.drop_duplicates()

# Faire des remplacements ou suppression de termes pour faciliter le matching
df["nom_clean"] = df["nom"].str.replace("Université", "University", case=False, regex=False)
df["nom_clean"] = df["nom_clean"].str.replace("Universidad", "University", case=False, regex=False)
df["nom_clean"] = df["nom_clean"].str.replace("Universita", "University", case=False, regex=False)
df["nom_clean"] = df["nom_clean"].str.replace("Universidade", "University", case=False, regex=False)
df["nom_clean"] = df["nom_clean"].str.replace("Universität", "University", case=False, regex=False)
df["nom_clean"] = df["nom_clean"].str.replace("&", "and", case=False, regex=False)
df["nom_clean"] = df["nom_clean"].str.replace("-", " ", case=False, regex=False)
df["nom_clean"] = df["nom_clean"].str.replace("/", " ", case=False, regex=False)

# Supprimer les mots inutiles : "de", "du", "de la", "d'", "of"
def retirer_termes_inutiles(texte):
    if pd.isna(texte):
        return ""
    texte = str(texte)
    texte = re.sub(r"\bde la\b", "", texte, flags=re.IGNORECASE)  # "de la"
    texte = re.sub(r"\bdu\b", "", texte, flags=re.IGNORECASE)     # "du"
    texte = re.sub(r"\bdi\b", "", texte, flags=re.IGNORECASE)     # "di"
    texte = re.sub(r"\bd[e']\b", "", texte, flags=re.IGNORECASE)  # "de" ou "d'"
    texte = re.sub(r"\bd’", "", texte, flags=re.IGNORECASE)       # apostrophe typographique
    texte = re.sub(r"\bof\b", "", texte, flags=re.IGNORECASE)     # "of"
    
    return texte.strip()

# Appliquer le nettoyage des termes avant le nettoyage global
df["nom_clean"] = df["nom_clean"].apply(retirer_termes_inutiles)

# Fonction de nettoyage (sluggify)
def nettoyer_texte(texte):
    if pd.isna(texte):
        return ""
    texte = str(texte)        # convertir en chaîne de caractères (et non en nombre)
    texte = unidecode(texte)  # enlève les accents
    texte = texte.lower()     # minuscule
    texte = re.sub(r"[^\w\s]", "", texte)  # enlève tout sauf lettres/chiffres (supprime ponctuation et autres, mais garde espace)
    texte = re.sub(r"\s+", " ", texte)      # réduit espaces multiples
    return texte

# Appliquer la fonction
df["nom_clean"] = df["nom_clean"].apply(nettoyer_texte)

# Ajouter une colonne "longueur" pour trier
df["longueur"] = df["nom_clean"].str.len()

# Trier par longueur décroissante
df = df.sort_values(by="longueur", ascending=False)

# Supprimer la colonne "longueur" si tu ne veux pas la garder
df = df.drop(columns=["longueur"])

# Sauvegarder dans un nouveau fichier Excel
df.to_excel("dictionnaire_clean.xlsx", index=False)

print("Nettoyage terminé. Fichier enregistré sous 'dictionnaire_clean.xlsx'")

Nettoyage terminé. Fichier enregistré sous 'dictionnaire_clean.xlsx'


In [5]:
import pandas as pd
from unidecode import unidecode
import re

# NETTOYAGE DES DONNÉES DES ARTICLES

# Charger le fichier Excel
fichier2 = "data.xlsx"
df = pd.read_excel(fichier2)

# Faire des remplacements ou suppression de termes pour faciliter le matching
df["author_affiliation_content_clean"] = df["author_affiliation_content"].str.replace("Université", "University", case=False, regex=False)
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].str.replace("Universidad", "University", case=False, regex=False)
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].str.replace("Universita", "University", case=False, regex=False)
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].str.replace("Universidade", "University", case=False, regex=False)
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].str.replace("Universität", "University", case=False, regex=False)
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].str.replace("&", "and", case=False, regex=False)
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].str.replace("-", " ", case=False, regex=False)
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].str.replace("/", " ", case=False, regex=False)

# Appliquer le nettoyage des termes avant le nettoyage global
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].apply(retirer_termes_inutiles)

# Appliquer le nettoyage globale
df["author_affiliation_content_clean"] = df["author_affiliation_content_clean"].apply(nettoyer_texte)

# Sauvegarder dans un nouveau fichier Excel
df.to_excel("data_clean.xlsx", index=False)

print("Nettoyage terminé. Fichier enregistré sous 'data_clean.xlsx'")

Nettoyage terminé. Fichier enregistré sous 'data_clean.xlsx'


In [ ]:
import pandas as pd
from unidecode import unidecode
import re

# FAIRE LES MATCHS

# Charger les deux fichiers nettoyés
df1 = pd.read_excel("data_clean.xlsx")
df2 = pd.read_excel("dictionnaire_clean.xlsx")

# Extraire la liste des alias nettoyés
liste_alias = df2["nom_clean"].tolist()

# Créer une fonction de recherche progressive
# Cherche à partir du premier mot de l'affiliation, si n'y a pas de match d'au moins 8 caractères, fait la recherche à partir du 2e mot, et ainsi de suite
# Si ne trouve rien, fait une recherche sans souci de position et ramène le plus long match trouvé

def trouver_match_progressif(affiliation, liste_alias):
    if pd.isna(affiliation):
        return None

    # On sépare l'affiliation en mots
    mots = affiliation.split()

    # Tester chaque sous-chaîne commençant à un mot différent (avec taille de 8 caractères minimum)
    for i in range(len(mots)):
        sous_chaine = " ".join(mots[i:])
        for org in liste_alias:
            if sous_chaine.startswith(org) and len(org) >= 8:
                return org  # premier alias trouvé avec la contrainte
                
    # Fallback : # Tester chaque sous-chaîne commençant à un mot différent (sans taille minimum)
    for i in range(len(mots)):
        sous_chaine = " ".join(mots[i:])
        for org in liste_alias:
            if sous_chaine.startswith(org):
                return org  # premier alias trouvé avec la contrainte
            
    # Si aucun match
    return None


# Appliquer la fonction à chaque ligne des données d'affiliation
df1["nom_trouve1"] = df1["author_affiliation_content_clean"].apply(
    lambda x: trouver_match_progressif(x, liste_alias))

# Faire une 1re jointure pour nom_trouve1
df1 = df1.merge(df2[["nom_clean", "ror"]], how="left", left_on="nom_trouve1", right_on="nom_clean")
df1 = df1.rename(columns={"ror": "ror1"})  # Renommer pour éviter collision

# Supprimer la colonne temporaire utilisée pour la jointure
df1 = df1.drop(columns=["nom_clean"])

# Supprimer les lignes doublons
df1 = df1.drop_duplicates()

# Sauvegarder dans un nouveau fichier Excel
df1.to_excel("results.xlsx", index=False)

print("Matching terminé. Résultat enregistré dans 'results.xlsx'")
